# Mise en place de la pipeline

Import libraires

In [1]:
from google.colab import files
import torch
import torch.nn as nn
from torchvision import models
from torch.utils.data import DataLoader
from tqdm import tqdm
import sys
import os

Clonage du repo git sur le colab

In [2]:
!git clone https://github.com/alextonon/Hackaton_abeilles_tigres

Cloning into 'Hackaton_abeilles_tigres'...
remote: Enumerating objects: 462, done.
remote: Counting objects: 100% (19/19), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 462 (delta 9), reused 13 (delta 5), pack-reused 443 (from 1)
Receiving objects: 100% (462/462), 219.35 MiB | 25.08 MiB/s, done.
Resolving deltas: 100% (234/234), done.
Updating files: 100% (71/71), done.


Téléchargement des données depuis kaggle

In [4]:
# 1. Upload du fichier kaggle.json (choisir le bon fichier sur l'ordinateur)
files.upload()

# 2. Création du dossier caché et déplacement du fichier
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/

# 3. Sécurisation du fichier (obligatoire pour l'API Kaggle)
!chmod 600 ~/.kaggle/kaggle.json

Saving kaggle.json to kaggle.json


In [5]:
#Téléchargement du fichier depuis kaggle
!kaggle competitions download -c reconnaissance-dabeilles-francaises

 99% 992M/999M [00:10<00:00, 29.4MB/s]
100% 999M/999M [00:10<00:00, 101MB/s] 


In [ ]:
#Dézipage du fichier téléchargé depuis kaggle
!unzip -q reconnaissance-dabeilles-francaises.zip -d Hackaton_abeilles_tigres

Le flux de sortie a été tronqué et ne contient que les 5000 dernières lignes.
  inflating: Hackaton_abeilles_tigres/data/train/Andrena denticulata/516cd2797cfa69981878b09ced74073fdbfdb5cf.jpg  
  inflating: Hackaton_abeilles_tigres/data/train/Andrena denticulata/51845ee19a8e0dea0122d9bfca55e3d834d51bf1.jpg  
  inflating: Hackaton_abeilles_tigres/data/train/Andrena denticulata/52175e5f6184fb1b688e3a45201938a67a18e5e1.jpg  
  inflating: Hackaton_abeilles_tigres/data/train/Andrena denticulata/52d2280252bf1ac35358d663a5db3da002f92df7.jpg  
  inflating: Hackaton_abeilles_tigres/data/train/Andrena denticulata/52d7a43e7978384d30aa4efabc7a84dcb6d4600a.jpg  
  inflating: Hackaton_abeilles_tigres/data/train/Andrena denticulata/5312ea1d192edb4b2445e1e97a1616021b9fb0e3.jpg  
  inflating: Hackaton_abeilles_tigres/data/train/Andrena denticulata/53e4917f1b805d05e5c1ecf62b68f8c6a13286b4.jpg  
  inflating: Hackaton_abeilles_tigres/data/train/Andrena denticulata/540baf72c0c59c24255732f5efd206d70d0d7b79.

In [7]:
#On se place dans le repo
%cd /content/Hackaton_abeilles_tigres/

/content/Hackaton_abeilles_tigres


In [8]:
#Déplacement des classe
!rm -rf data/train/Andrena\ dilleri data/train/Andrena\ pinguis data/train/Andrena\ prodigiosa
!mv processed_classes/* Hackaton_abeilles_tigres/data/train

mv: target 'Hackaton_abeilles_tigres/data/train' is not a directory


Création du csv train_corrected

In [9]:
!python3 lib/data/train_csv.py

Done ! 7982 lines written in data/train_corrected.csv


# Entraînement du modèle

In [10]:
sys.path.append(os.path.abspath(".."))
from lib.data.dataset import BeeDataset
from lib.data.preprocessing import TorchPreprocessor
from lib.data.train_val_split import train_val_split
from lib.data.preprocessing import TorchPreprocessor
from lib.data.data_augmentation import data_augmented_loader

In [11]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BATCH_SIZE = 32
EPOCHS = 10
LR = 1e-3
num_classes = 50

notebook_dir = os.getcwd()
data_dir = os.path.abspath(os.path.join(notebook_dir, "data"))
print(data_dir)

/content/Hackaton_abeilles_tigres/data


## 1. Preprocessing

In [12]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# preprocessor = TorchPreprocessor(
#     normalize=True,
#     mean = IMAGENET_MEAN,
#     std = IMAGENET_STD,
#     resize_method="pad",
#     target_size=(224, 224),
# )

# train_dataset, val_dataset = train_val_split(train_transform=preprocessor, val_transform=preprocessor)

In [13]:
import lib.data.preprocessing as prep
print(prep.__file__)

/content/Hackaton_abeilles_tigres/lib/data/preprocessing.py


In [14]:
%cd /content/Hackaton_abeilles_tigres/lib


train_loader, val_loader = data_augmented_loader(IMAGENET_MEAN, IMAGENET_STD, target_size=(224, 224), batch_size=BATCH_SIZE)

/content/Hackaton_abeilles_tigres/lib
Train prêt : 6403 images (avec augmentation ciblée)
Val prête  : 1579 images (sans augmentation)


## 2. Modèle

In [15]:
def create_model(num_classes: int) -> nn.Module:
    model = models.efficientnet_b0(weights="IMAGENET1K_V1") #mettre b3 si ca marche
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.4),
        nn.Linear(in_features, num_classes),
    )
    return model

model = create_model(num_classes)
model.to(DEVICE)

criterion = nn.CrossEntropyLoss()

# --- Variante ---
# pip install torchmetrics
# from torchmetrics.classification import MulticlassFocalLoss
# criterion = MulticlassFocalLoss(num_classes=num_classes, alpha=0.25, gamma=2.0)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=1e-4
)


Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 152MB/s]


## 3 F1-score

In [16]:
import numpy as np

def compute_f1(all_labels, all_preds, num_classes):
    f1_per_class = []

    for cls in range(num_classes):
        # True Positives
        TP = np.sum((np.array(all_preds) == cls) & (np.array(all_labels) == cls))
        # False Positives
        FP = np.sum((np.array(all_preds) == cls) & (np.array(all_labels) != cls))
        # False Negatives
        FN = np.sum((np.array(all_preds) != cls) & (np.array(all_labels) == cls))

        precision = TP / (TP + FP) if (TP + FP) > 0 else 0
        recall    = TP / (TP + FN) if (TP + FN) > 0 else 0

        f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        f1_per_class.append(f1)

    # F1 macro : moyenne des classes
    f1_macro = np.mean(f1_per_class)
    return f1_macro, f1_per_class

## 4. Fonctions de training et validation

In [17]:
def train_one_epoch():
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    all_preds = []
    all_labels = []

    for images, labels in tqdm(train_loader):
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    # 🔹 Calcul F1 avec ta fonction existante
    f1_macro, f1_per_class = compute_f1(all_labels, all_preds, num_classes)

    # 🔹 Affichage
    # print(f"Train F1 macro: {f1_macro:.4f}")

    return total_loss / len(train_loader), correct / total, f1_macro, f1_per_class


In [18]:
import torch

def validate():
    model.eval()
    total_loss = 0

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)

            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item()

            preds = outputs.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # Calcul F1 manuel par classe
    f1_per_class = []
    for cls in range(num_classes):
        TP = np.sum((np.array(all_preds) == cls) & (np.array(all_labels) == cls))
        FP = np.sum((np.array(all_preds) == cls) & (np.array(all_labels) != cls))
        FN = np.sum((np.array(all_preds) != cls) & (np.array(all_labels) == cls))

        precision = TP / (TP + FP) if (TP + FP) > 0 else 0
        recall    = TP / (TP + FN) if (TP + FN) > 0 else 0

        f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        f1_per_class.append(f1)

    f1_macro = np.mean(f1_per_class)

    return total_loss / len(val_loader), f1_macro, f1_per_class

## Entraînement

Vérif des labels

In [19]:
# all_labels = [label for _, label in train_dataset]
# print("Label min:", min(all_labels))
# print("Label max:", max(all_labels))
# print("Nombre de classes uniques:", len(set(all_labels)))

# # Récupérer tous les labels uniques triés
# all_labels = sorted(set(label for _, label in train_dataset.samples))
# label_to_index = {label: i for i, label in enumerate(all_labels)}

# # Remapper les labels dans le dataset
# # for i, (path, label) in enumerate(train_dataset.samples):
# #     train_dataset.samples[i] = (path, label_to_index[label])

Entraînement

In [21]:
import csv
best_f1 = 0.0
best_model_path = "best_model.pth"

# Configuration du logging CSV
csv_file = "training_log.csv"
fieldnames = ['epoch', 'train_loss', 'train_acc', 'train_f1_macro', 'val_loss', 'val_f1_macro']

# Initialisation du fichier (écrase le précédent s'il existe)
with open(csv_file, mode='w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()

for epoch in range(EPOCHS):
    train_loss, train_acc, train_f1_macro, train_f1_per_class = train_one_epoch()
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Train F1 Macro: {train_f1_macro:.4f}")
    print(f"Train F1 per class: {train_f1_per_class}")

    val_loss, val_f1_macro, val_f1_per_class = validate()
    print(f"Val   Loss: {val_loss:.4f}")
    print(f"Val   F1 Macro: {val_f1_macro:.4f}")
    print(f"Val   F1 per class: {val_f1_per_class}")

    with open(csv_file, mode='a', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writerow({
            'epoch': epoch + 1,
            'train_loss': train_loss,
            'train_acc': train_acc,
            'train_f1_macro': train_f1_macro,
            'val_loss': val_loss,
            'val_f1_macro': val_f1_macro
        })

    # 🔹 Sauvegarde du meilleur modèle
    if val_f1_macro > best_f1:
        best_f1 = val_f1_macro
        torch.save(model.state_dict(), best_model_path)
        print(f" Nouveau meilleur modèle sauvegardé ! F1 Macro = {best_f1:.4f}")

100%|██████████| 201/201 [00:49<00:00,  4.07it/s]



Epoch 1/10
Train Loss: 0.2801 | Train Acc: 0.9139
Train F1 Macro: 0.8581
Train F1 per class: [0, np.float64(1.0), np.float64(0.7288135593220338), np.float64(0.9825783972125435), np.float64(0.9612403100775194), np.float64(0.8050847457627118), np.float64(0.9652777777777778), np.float64(0.9831932773109243), np.float64(0.9907407407407407), np.float64(0.9555555555555555), np.float64(0.9890909090909091), np.float64(0.9534883720930233), np.float64(0.7482993197278912), np.float64(0.9427609427609427), np.float64(0.9212598425196851), np.float64(0.9365079365079365), np.float64(0.9868421052631579), np.float64(0.842911877394636), np.float64(0.8529411764705882), np.float64(0.883720930232558), np.float64(0.9955555555555555), np.float64(0.9357142857142857), np.float64(0.8339483394833948), np.float64(0.9489051094890512), np.float64(0.7257383966244726), np.float64(0.8239436619718309), 0, np.float64(0.996415770609319), np.float64(0.8242811501597443), np.float64(0.9625000000000001), np.float64(1.0), np.f

100%|██████████| 201/201 [00:51<00:00,  3.93it/s]



Epoch 2/10
Train Loss: 0.2662 | Train Acc: 0.9218
Train F1 Macro: 0.8653
Train F1 per class: [0, np.float64(0.9754385964912281), np.float64(0.7692307692307693), np.float64(0.9690721649484535), np.float64(0.965034965034965), np.float64(0.7589285714285715), np.float64(0.9632352941176471), np.float64(0.9880478087649401), np.float64(0.9822064056939502), np.float64(0.9415807560137458), np.float64(0.9822064056939502), np.float64(0.9593495934959351), np.float64(0.7607843137254902), np.float64(0.9530685920577617), np.float64(0.900709219858156), np.float64(0.9460580912863071), np.float64(0.9844961240310077), np.float64(0.9182879377431907), np.float64(0.8175182481751824), np.float64(0.9492753623188407), np.float64(0.9965156794425087), np.float64(0.9291338582677164), np.float64(0.8617886178861789), np.float64(0.9599999999999999), np.float64(0.8453608247422679), np.float64(0.8222222222222222), 0, np.float64(0.9868421052631579), np.float64(0.8376623376623378), np.float64(0.9885057471264368), np.fl

 69%|██████▉   | 139/201 [00:33<00:15,  4.11it/s]


KeyboardInterrupt: 